# STR Factor Backtest with Tushare

该 Notebook 演示如何使用 **Tushare** 获取 A 股日频行情与换手率数据，构造 20 日滚动标准差的 STR 因子，进行市值中性化，并完成 T+1 十等分组回测、IC/ICIR 评估与可视化。代码注释均为中文，图表标题和图例保持英文。生成的 PNG 图片会保存到 `outputs/images` 目录以便归档。

In [ ]:
# 导入所需库（中文注释说明用途，图表相关标题将保持英文）
import os
import time
import math
import json
from pathlib import Path
from typing import List, Dict, Optional
import datetime as dt

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tushare as ts
from sklearn.linear_model import LinearRegression

# Matplotlib 全局样式配置
plt.style.use("seaborn-v0_8")
sns.set_context("talk")

# 关闭科学计数法的显示，便于阅读
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")


In [ ]:
# 路径与全局参数配置（可根据需要修改）
# Tushare 需要设置自己的 TOKEN，请在运行前替换为有效的字符串
TUSHARE_TOKEN = "YOUR_TUSHARE_TOKEN"

# 回测区间（形如 20180101）
START_DATE = "20180101"
END_DATE = "20241231"

# 请求节奏控制：默认每次请求后等待 10 秒，降低被封禁 IP 的风险
REQUEST_INTERVAL_SECS = 10.0
MAX_RETRY = 3

# 最多处理的股票数量（调试时可设置较小值，例如 50；正式跑全量可设 None 或较大值）
MAX_STOCKS = 50

# 输出目录配置（所有 PNG 图片会保存到此处）
OUTPUT_DIR = Path("outputs")
IMAGE_DIR = OUTPUT_DIR / "images"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

print("输出目录已准备：", IMAGE_DIR.resolve())


In [ ]:
# 工具函数：初始化 Tushare Pro 对象
# 该函数封装 TOKEN 设置并返回 pro 句柄

def init_tushare(token: str = TUSHARE_TOKEN):
    """初始化并返回 Tushare Pro 接口对象"""
    ts.set_token(token)
    pro = ts.pro_api()
    return pro


In [ ]:
# 工具函数：带节奏与重试的请求封装
# 作用：统一控制请求间隔与重试逻辑，减少网络抖动或限流导致的错误

def throttled_call(func, *, sleep_seconds: float = REQUEST_INTERVAL_SECS, max_retry: int = MAX_RETRY, **kwargs):
    """对任意 Tushare 接口函数添加重试与等待逻辑"""
    last_error = None
    for attempt in range(1, max_retry + 1):
        try:
            result = func(**kwargs)
            # 每次成功请求后等待指定秒数，降低请求频率
            time.sleep(sleep_seconds)
            return result, None
        except Exception as exc:  # noqa: BLE001
            last_error = exc
            print(f"[警告] 第 {attempt} 次调用 {func.__name__} 失败：{exc}")
            if attempt < max_retry:
                time.sleep(sleep_seconds)
    print(f"[错误] 多次尝试调用 {func.__name__} 仍失败，返回空数据")
    return pd.DataFrame(), last_error


In [ ]:
# 获取股票基础列表，并过滤 ST、新股等
# 只保留主板/创业板等正常交易股票，同时可限制最大数量加速调试

def fetch_stock_list(pro, max_stocks: Optional[int] = MAX_STOCKS) -> pd.DataFrame:
    """获取 A 股股票列表，过滤 ST，并按 ts_code 排序"""
    df, err = throttled_call(
        pro.stock_basic,
        exchange="",
        list_status="L",
        fields="ts_code,symbol,name,area,industry,list_date,market,list_status"
    )
    if isinstance(df, pd.DataFrame) and not df.empty:
        # 过滤 ST 股票：名称包含 ST 或 *ST
        mask_st = df["name"].str.contains("ST", case=False)
        clean_df = df.loc[~mask_st].copy()
        # 按上市日期过滤新股（例如上市未满 120 天）
        cutoff_date = (dt.datetime.today() - dt.timedelta(days=120)).strftime("%Y%m%d")
        clean_df = clean_df.loc[clean_df["list_date"] <= cutoff_date]
        clean_df = clean_df.sort_values("ts_code")
        if max_stocks is not None:
            clean_df = clean_df.head(max_stocks)
        print(f"可投资股票数量：{len(clean_df)}")
        return clean_df
    print("[错误] 股票列表为空或获取失败")
    return pd.DataFrame()


In [ ]:
# 针对单只股票拉取价格与换手率数据，并进行合并
# 使用 pro_bar 获取前复权收盘价与成交量，用 daily_basic 获取换手率与流通市值

def fetch_single_stock_data(pro, ts_code: str, start_date: str = START_DATE, end_date: str = END_DATE) -> pd.DataFrame:
    """拉取单个股票的日度行情与换手率、流通市值数据"""
    price_df, _ = throttled_call(
        pro.pro_bar,
        ts_code=ts_code,
        start_date=start_date,
        end_date=end_date,
        adj="qfq"
    )
    basic_df, _ = throttled_call(
        pro.daily_basic,
        ts_code=ts_code,
        start_date=start_date,
        end_date=end_date,
        fields="ts_code,trade_date,turnover_rate,free_share,total_mv,circ_mv"
    )
    if isinstance(price_df, pd.DataFrame) and not price_df.empty:
        price_df = price_df[["ts_code", "trade_date", "close"]].copy()
    else:
        return pd.DataFrame()

    if isinstance(basic_df, pd.DataFrame) and not basic_df.empty:
        basic_df = basic_df[["ts_code", "trade_date", "turnover_rate", "circ_mv"]].copy()
    else:
        basic_df = pd.DataFrame(columns=["ts_code", "trade_date", "turnover_rate", "circ_mv"])

    merged = pd.merge(price_df, basic_df, on=["ts_code", "trade_date"], how="left")
    merged["trade_date"] = pd.to_datetime(merged["trade_date"])
    merged = merged.sort_values("trade_date")
    merged = merged.rename(columns={"close": "close_qfq", "turnover_rate": "turnover"})
    return merged


In [ ]:
# 构建面板数据：循环股票列表，逐一抓取并存储
# 注意：每只股票的请求之间已经在 throttled_call 中加入等待，这里无需额外 sleep

def prepare_panel_data(pro, stock_list: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    """批量获取股票数据，返回 {ts_code: DataFrame} 字典"""
    panel = {}
    for idx, row in enumerate(stock_list.itertuples(), start=1):
        ts_code = row.ts_code
        print(f"[{idx}/{len(stock_list)}] 拉取 {ts_code} ...")
        df = fetch_single_stock_data(pro, ts_code=ts_code)
        if not df.empty:
            panel[ts_code] = df
        else:
            print(f"[提示] {ts_code} 数据为空，已跳过")
    print(f"完成拉取，共 {len(panel)} 只股票")
    return panel


In [ ]:
# 因子计算：20 日滚动标准差 + 市值中性化
# 先计算原始因子 STR_raw，再横截面回归对数流通市值进行中性化

def compute_str_factor(panel: Dict[str, pd.DataFrame], window: int = 20) -> pd.DataFrame:
    """基于面板数据计算市值中性化后的 STR 因子"""
    factor_frames = []
    for ts_code, df in panel.items():
        tmp = df.copy()
        tmp["str_raw"] = tmp["turnover"].rolling(window=window, min_periods=window).std()
        tmp["log_circ_mv"] = np.log(tmp["circ_mv"])
        tmp["ts_code"] = ts_code
        factor_frames.append(tmp[["trade_date", "ts_code", "str_raw", "log_circ_mv", "close_qfq"]])
    factor_df = pd.concat(factor_frames, ignore_index=True)

    # 横截面中性化：按日期分组回归 str_raw ~ log_circ_mv
    neutral_list = []
    for trade_date, grp in factor_df.groupby("trade_date"):
        grp = grp.dropna(subset=["str_raw", "log_circ_mv"])
        if len(grp) < 5:
            continue
        X = grp[["log_circ_mv"]].values
        y = grp["str_raw"].values
        model = LinearRegression()
        model.fit(X, y)
        residuals = y - model.predict(X)
        temp = grp.copy()
        temp["str"] = residuals
        neutral_list.append(temp)
    neutral_df = pd.concat(neutral_list, ignore_index=True)
    return neutral_df


In [ ]:
# 构建 T+1 十等分组收益
# 在交易日 t 使用当日因子值排序，第二天计算等权收益

def build_group_returns(factor_df: pd.DataFrame, group_num: int = 10) -> pd.DataFrame:
    """根据因子值分组并计算下一交易日的等权收益"""
    # 为方便计算，将数据按日期、ts_code 排序
    factor_df = factor_df.sort_values(["trade_date", "ts_code"])

    # 计算次日收益率 r_{t+1}
    factor_df["next_close"] = factor_df.groupby("ts_code")["close_qfq"].shift(-1)
    factor_df["ret_t1"] = factor_df["next_close"] / factor_df["close_qfq"] - 1

    results = []
    for trade_date, grp in factor_df.groupby("trade_date"):
        grp = grp.dropna(subset=["str", "ret_t1"])
        if len(grp) < group_num:
            continue
        grp = grp.sort_values("str")
        grp["group"] = pd.qcut(grp["str"], group_num, labels=False) + 1
        grouped = grp.groupby("group")["ret_t1"].mean().reset_index()
        grouped["trade_date"] = trade_date
        results.append(grouped)
    group_ret = pd.concat(results, ignore_index=True)
    pivot = group_ret.pivot(index="trade_date", columns="group", values="ret_t1")
    pivot.columns = [f"G{c}" for c in pivot.columns]
    pivot = pivot.sort_index()
    return pivot


In [ ]:
# 计算净值曲线、年化收益、波动率、最大回撤等指标

def calc_nav_and_stats(group_ret: pd.DataFrame) -> Dict[str, Dict[str, float]]:
    """根据分组日度收益计算净值与关键风险收益指标"""
    nav = (1 + group_ret).cumprod()
    stats = {}
    ann_factor = 252
    for col in group_ret.columns:
        daily_ret = group_ret[col].dropna()
        if daily_ret.empty:
            continue
        mean_ret = daily_ret.mean()
        vol = daily_ret.std()
        ann_ret = mean_ret * ann_factor
        ann_vol = vol * math.sqrt(ann_factor)
        nav_series = nav[col]
        peak = nav_series.cummax()
        dd = 1 - nav_series / peak
        max_dd = dd.max()
        stats[col] = {
            "ann_ret": ann_ret,
            "ann_vol": ann_vol,
            "max_dd": max_dd
        }
    return nav, stats


In [ ]:
# 计算 IC 与 ICIR

def calc_ic_icir(factor_df: pd.DataFrame) -> pd.DataFrame:
    """计算日度 IC 及其统计指标"""
    ic_list = []
    factor_df = factor_df.sort_values(["ts_code", "trade_date"])
    factor_df["next_ret"] = factor_df.groupby("ts_code")["close_qfq"].pct_change(-1)
    for trade_date, grp in factor_df.groupby("trade_date"):
        grp = grp.dropna(subset=["str", "next_ret"])
        if len(grp) < 5:
            continue
        ic = grp[["str", "next_ret"]].corr().iloc[0, 1]
        ic_list.append({"trade_date": trade_date, "ic": ic})
    ic_df = pd.DataFrame(ic_list).sort_values("trade_date")
    ic_mean = ic_df["ic"].mean()
    ic_std = ic_df["ic"].std(ddof=1)
    icir = ic_mean / ic_std * math.sqrt(252) if ic_std not in (0, np.nan) else np.nan
    summary = pd.DataFrame({"ic_mean": [ic_mean], "ic_std": [ic_std], "icir": [icir]})
    return ic_df, summary


In [ ]:
# 画图工具：净值曲线与 IC 时间序列
# 图表标题、图例均使用英文；保存 PNG 到 IMAGE_DIR

def plot_nav(nav: pd.DataFrame, filename: str = "nav_curves.png"):
    """绘制分组净值曲线并保存"""
    plt.figure(figsize=(12, 6))
    for col in nav.columns:
        plt.plot(nav.index, nav[col], label=col)
    plt.title("Group NAV Curves (Multiplicative)")
    plt.xlabel("Trade Date")
    plt.ylabel("Net Asset Value")
    plt.legend(title="Group")
    plt.tight_layout()
    path = IMAGE_DIR / filename
    plt.savefig(path, dpi=150, format="png")
    plt.show()
    print(f"净值曲线已保存：{path}")


def plot_ic_series(ic_df: pd.DataFrame, filename: str = "ic_series.png"):
    """绘制 IC 时间序列并保存"""
    plt.figure(figsize=(12, 6))
    plt.plot(ic_df["trade_date"], ic_df["ic"], label="Daily IC")
    plt.axhline(0, color="black", linestyle="--", linewidth=1)
    plt.title("Daily IC Series")
    plt.xlabel("Trade Date")
    plt.ylabel("IC")
    plt.legend()
    plt.tight_layout()
    path = IMAGE_DIR / filename
    plt.savefig(path, dpi=150, format="png")
    plt.show()
    print(f"IC 序列已保存：{path}")


In [ ]:
# 主流程示例（为避免 Notebook 在无网络环境阻塞，默认不自动运行，可根据需要手动执行）
# 运行步骤：
# 1. 替换有效的 TUSHARE_TOKEN。
# 2. 取消注释下方的调用并逐步执行。
# 3. 查看 outputs/images 下的 PNG 结果。

# pro = init_tushare()
# stock_list = fetch_stock_list(pro, max_stocks=MAX_STOCKS)
# panel = prepare_panel_data(pro, stock_list)
# factor_df = compute_str_factor(panel, window=20)
# group_ret = build_group_returns(factor_df, group_num=10)
# nav, stats = calc_nav_and_stats(group_ret)
# ic_df, ic_summary = calc_ic_icir(factor_df)

# print("分组关键指标：
", json.dumps(stats, indent=2, ensure_ascii=False))
# print("IC/ICIR 摘要：
", ic_summary)

# plot_nav(nav)
# plot_ic_series(ic_df)
